# imports

In [135]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os

# connection and load

In [136]:
db_path = '../../database/financial_data.duckdb'
conn = duckdb.connect(db_path)

In [137]:
ASSET = 'BTC'
INTERVAL = '4h'

# Load from Gold Layer

In [138]:
query = f"""
    SELECT * 
    FROM gold_crypto_features 
    WHERE asset_symbol = '{ASSET}' 
    AND interval = '{INTERVAL}' 
    ORDER BY date ASC
"""

print(f"Loading {ASSET} {INTERVAL} from gold_crypto_features...")
df = conn.execute(query).df()
print(f"Loaded: {df.shape}")

Loading BTC 4h from gold_crypto_features...
Loaded: (13274, 52)


# datetime index

In [139]:
df.head(3)

,asset_symbol,asset_class,exchange,interval,date,open,high,low,close,volume,...,log_returns,hl_ratio,close_position,prev_close,prev_volume,prev_high,prev_low,turnover,open_interest,funding_rate
0,BTC,Crypto,Bybit,4h,2020-04-27 12:00:00,7713.5,7722.5,7624.0,7669.5,1068.971,...,-0.005721,0.012843,0.461929,7713.5,832.093,7736.0,7666.0,8.198473e+06,NaN,NaN
1,BTC,Crypto,Bybit,4h,2020-04-27 16:00:00,7669.5,7718.5,7642.0,7700.5,588.447,...,0.004034,0.009934,0.764706,7669.5,1068.971,7722.5,7624.0,4.531336e+06,NaN,NaN
2,BTC,Crypto,Bybit,4h,2020-04-27 20:00:00,7700.5,7777.5,7686.0,7772.5,838.893,...,0.009307,0.011772,0.945355,7700.5,588.447,7718.5,7642.0,6.520296e+06,NaN,NaN


In [140]:
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
print(f"Index set to datetime. Date range: {df.index.min()} → {df.index.max()}")

Index set to datetime. Date range: 2020-04-27 12:00:00 → 2026-05-18 16:00:00


In [141]:
df.head(3)

,asset_symbol,asset_class,exchange,interval,open,high,low,close,volume,daily_volatility,...,log_returns,hl_ratio,close_position,prev_close,prev_volume,prev_high,prev_low,turnover,open_interest,funding_rate
date,,,,,,,,,,,,,,,,,,,,,
2020-04-27 12:00:00,BTC,Crypto,Bybit,4h,7713.5,7722.5,7624.0,7669.5,1068.971,98.5,...,-0.005721,0.012843,0.461929,7713.5,832.093,7736.0,7666.0,8.198473e+06,NaN,NaN
2020-04-27 16:00:00,BTC,Crypto,Bybit,4h,7669.5,7718.5,7642.0,7700.5,588.447,76.5,...,0.004034,0.009934,0.764706,7669.5,1068.971,7722.5,7624.0,4.531336e+06,NaN,NaN
2020-04-27 20:00:00,BTC,Crypto,Bybit,4h,7700.5,7777.5,7686.0,7772.5,838.893,91.5,...,0.009307,0.011772,0.945355,7700.5,588.447,7718.5,7642.0,6.520296e+06,NaN,NaN


In [142]:
print(f"Shape: {df.shape}")
df.info(verbose=True, show_counts=True)

Shape: (13274, 51)
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 13274 entries, 2020-04-27 12:00:00 to 2026-05-18 16:00:00
Data columns (total 51 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   asset_symbol      13274 non-null  object 
 1   asset_class       13274 non-null  object 
 2   exchange          13274 non-null  object 
 3   interval          13274 non-null  object 
 4   open              13274 non-null  float64
 5   high              13274 non-null  float64
 6   low               13274 non-null  float64
 7   close             13274 non-null  float64
 8   volume            13274 non-null  float64
 9   daily_volatility  13274 non-null  float64
 10  sma_7             13274 non-null  float64
 11  sma_30            13274 non-null  float64
 12  rsi_14            13274 non-null  float64
 13  macd              13274 non-null  float64
 14  macd_signal       13274 non-null  float64
 15  macd_histogram    13274 non-null 

# drop metadata cols

In [143]:
metadata_cols = ['asset_symbol', 'asset_class', 'exchange', 'interval']
df = df.drop(columns=metadata_cols, errors='ignore')
print(f"Columns remaining: {len(df.columns)}")

Columns remaining: 47


# check  specific cols (OI, Turnover, Funding Rate)

In [144]:
new_cols = ['turnover', 'open_interest', 'funding_rate']
available_new = [c for c in new_cols if c in df.columns]
missing_new = [c for c in new_cols if c not in df.columns]

print(f"Columns available: {available_new}")
print(f"Columns MISSING:    {missing_new}")

Columns available: ['turnover', 'open_interest', 'funding_rate']
Columns MISSING:    []


#  coverage analysis

In [145]:
for c in available_new:
    coverage = (1 - df[c].isna().mean()) * 100
    print(f"  {c}: {coverage:.1f}% non-null ({df[c].notna().sum():,} / {len(df):,} rows)")

  turnover: 100.0% non-null (13,274 / 13,274 rows)
  open_interest: 0.0% non-null (0 / 13,274 rows)
  funding_rate: 3.0% non-null (399 / 13,274 rows)


# drop funding_rate (unusable ~3% coverage)

In [146]:
if 'funding_rate' in df.columns:
    df = df.drop(columns=['funding_rate'])
    print("Dropped funding_rate — not suitable for model training")
else:
    print("funding_rate not in cols, nothing to drop")

Dropped funding_rate — not suitable for model training


# open interest feature engineering

In [147]:
if 'open_interest' in df.columns:
    oi_null_pct = df['open_interest'].isna().mean() * 100
    print(f"open_interest: {oi_null_pct:.1f}% NaN")
    
    if df['open_interest'].isna().all():
        # 100% NaN — drop the column, skip feature engineering
        df = df.drop(columns=['open_interest'])
        print("⚠️ open_interest is 100% NULL — dropped column, skipping OI features")
    else:
        df['oi_change_1p']  = df['open_interest'].pct_change(1)
        df['oi_change_5p']  = df['open_interest'].pct_change(5)
        df['oi_change_20p'] = df['open_interest'].pct_change(20)
        
        oi_roll_mean = df['open_interest'].rolling(50).mean()
        oi_roll_std  = df['open_interest'].rolling(50).std()
        df['oi_zscore_50'] = (df['open_interest'] - oi_roll_mean) / oi_roll_std
        
        df = df.drop(columns=['open_interest'])
        
        print("OI features engineered: oi_change_1p, oi_change_5p, oi_change_20p, oi_zscore_50")
        print(f"  oi_change_1p  NaN: {df['oi_change_1p'].isna().sum()} rows")
        print(f"  oi_change_5p  NaN: {df['oi_change_5p'].isna().sum()} rows")
        print(f"  oi_change_20p NaN: {df['oi_change_20p'].isna().sum()} rows")
        print(f"  oi_zscore_50  NaN: {df['oi_zscore_50'].isna().sum()} rows")

open_interest: 100.0% NaN
⚠️ open_interest is 100% NULL — dropped column, skipping OI features


# turnover feature engineering

In [148]:
if 'turnover' in df.columns:
    df['turnover_ratio'] = df['turnover'] / df['volume']
    df['turnover_change_1p'] = df['turnover'].pct_change(1)
    df['turnover_change_5p'] = df['turnover'].pct_change(5)
    
    tv_roll_mean = df['turnover_ratio'].rolling(50).mean()
    tv_roll_std  = df['turnover_ratio'].rolling(50).std()
    df['turnover_ratio_zscore_50'] = (df['turnover_ratio'] - tv_roll_mean) / tv_roll_std
    
    df = df.drop(columns=['turnover'])
    
    print("Turnover features engineered: turnover_ratio, turnover_change_1p, turnover_change_5p, turnover_ratio_zscore_50")
    print(f"  turnover_ratio          NaN: {df['turnover_ratio'].isna().sum()} rows")
    print(f"  turnover_change_1p      NaN: {df['turnover_change_1p'].isna().sum()} rows")
    print(f"  turnover_change_5p      NaN: {df['turnover_change_5p'].isna().sum()} rows")
    print(f"  turnover_ratio_zscore_50 NaN: {df['turnover_ratio_zscore_50'].isna().sum()} rows")

Turnover features engineered: turnover_ratio, turnover_change_1p, turnover_change_5p, turnover_ratio_zscore_50
  turnover_ratio          NaN: 0 rows
  turnover_change_1p      NaN: 1 rows
  turnover_change_5p      NaN: 5 rows
  turnover_ratio_zscore_50 NaN: 49 rows


# native target 4h (1-period forward return!)

In [149]:
df['target_4h'] = df['close'].pct_change(1).shift(-1)

# drop target nan the last row 

In [150]:
df = df.dropna(subset=['target_4h'])
print(f"Shape after dropping target NaN: {df.shape}")

Shape after dropping target NaN: (13273, 49)


In [151]:
df[['close', 'target_4h']].tail(6)

,close,target_4h
date,,
2026-05-17 16:00:00,78370.0,-0.012168
2026-05-17 20:00:00,77416.4,-0.006665
2026-05-18 00:00:00,76900.4,0.001856
2026-05-18 04:00:00,77043.1,0.002841
2026-05-18 08:00:00,77262.0,-0.011400
2026-05-18 12:00:00,76381.2,-0.000055


# drop remaining nan

In [152]:
before = len(df)
df = df.dropna()
after = len(df)

print(f"Data shape after dropping all NaN: {df.shape}")
print(f"Rows dropped: {before - after} ({(before - after) / before * 100:.1f}%)")
print(f"Total columns: {len(df.columns)} (features + target_4h + target_direction)")

Data shape after dropping all NaN: (13224, 49)
Rows dropped: 49 (0.4%)
Total columns: 49 (features + target_4h + target_direction)


# create target_direction and class balance

In [153]:
df['target_direction'] = (df['target_4h'] > 0).astype(int)

balance = df['target_direction'].value_counts(normalize=True) * 100
print("Class Balance (native 4h target):")
print(balance)

Class Balance (native 4h target):
target_direction
1    51.225045
0    48.774955
Name: proportion, dtype: float64


# stationarity transform: MAs → distance from price (%)

In [154]:
df['sma_7_dist']   = df['close'] / df['sma_7']   - 1
df['sma_30_dist']  = df['close'] / df['sma_30']  - 1
df['sma_50_dist']  = df['close'] / df['sma_50']  - 1
df['sma_100_dist'] = df['close'] / df['sma_100'] - 1
df['sma_200_dist'] = df['close'] / df['sma_200'] - 1
df['ema_12_dist']  = df['close'] / df['ema_12']  - 1
df['ema_26_dist']  = df['close'] / df['ema_26']  - 1
df['ema_50_dist']  = df['close'] / df['ema_50']  - 1
df['ema_200_dist'] = df['close'] / df['ema_200'] - 1
df['vwap_dist']    = df['close'] / df['vwap']    - 1

print("MA distances created (close/MA - 1)")

MA distances created (close/MA - 1)


# stationarity transform: MACD/ATR/Vol → % of price

In [155]:
df['macd_pct']      = df['macd']            / df['close']
df['macd_sig_pct']  = df['macd_signal']     / df['close']
df['macd_hist_pct'] = df['macd_histogram']  / df['close']
df['atr_pct']       = df['atr_14']          / df['close']
df['volatility_pct']= df['daily_volatility'] / df['close']

print("MACD/ATR/Vol scaled as % of price")

MACD/ATR/Vol scaled as % of price


# drop non stationary cols

In [156]:
non_stationary_cols = [
    'open', 'high', 'low', 'close', 
    'sma_7', 'sma_30', 'sma_50', 'sma_100', 'sma_200',
    'ema_12', 'ema_26', 'ema_50', 'ema_200', 
    'vwap', 'prev_close', 'prev_high', 'prev_low',
    'macd', 'macd_signal', 'macd_histogram', 
    'atr_14', 'daily_volatility',
    'bb_upper', 'bb_middle', 'bb_lower', 'bb_width', 
    'volume', 'prev_volume', 'volume_sma_20', 
    'obv' 
]

df = df.drop(columns=non_stationary_cols, errors='ignore')
print(f"Dropped {len(non_stationary_cols)} non-stationary columns")
print(f"Columns remaining: {len(df.columns)}")

Dropped 30 non-stationary columns
Columns remaining: 35


# final cleanup

In [157]:
df = df.dropna()
print(f"Final data shape for ML: {df.shape}")

Final data shape for ML: (13224, 35)


# featureslist

In [158]:
all_features = [c for c in df.columns if c not in ['target_4h', 'target_direction']]
print(f"Features ({len(all_features)}):")
for i, f in enumerate(all_features):
    print(f"  {i+1:2d}. {f}")

Features (33):
   1. rsi_14
   2. roc_10
   3. roc_20
   4. stoch_k
   5. stoch_d
   6. bb_percentage
   7. volume_ratio
   8. returns_1p
   9. returns_5p
  10. returns_10p
  11. returns_20p
  12. log_returns
  13. hl_ratio
  14. close_position
  15. turnover_ratio
  16. turnover_change_1p
  17. turnover_change_5p
  18. turnover_ratio_zscore_50
  19. sma_7_dist
  20. sma_30_dist
  21. sma_50_dist
  22. sma_100_dist
  23. sma_200_dist
  24. ema_12_dist
  25. ema_26_dist
  26. ema_50_dist
  27. ema_200_dist
  28. vwap_dist
  29. macd_pct
  30. macd_sig_pct
  31. macd_hist_pct
  32. atr_pct
  33. volatility_pct


# 80/20 chronological split

In [159]:
split_idx = int(len(df) * 0.80)
train_df = df.iloc[:split_idx].copy()
test_df  = df.iloc[split_idx:].copy()

print(f"Training: {train_df.index.min()} → {train_df.index.max()}  ({len(train_df):,} rows)")
print(f"Testing:  {test_df.index.min()} → {test_df.index.max()}  ({len(test_df):,} rows)")

Training: 2020-05-05 16:00:00 → 2025-03-03 16:00:00  (10,579 rows)
Testing:  2025-03-03 20:00:00 → 2026-05-18 12:00:00  (2,645 rows)


# feature set

In [160]:
features = [col for col in train_df.columns if col not in ['target_4h', 'target_direction']]
print(f"{len(features)} features for model input")

33 features for model input


#  StandardScaler (fit on train, transform on test)

In [161]:
scaler = StandardScaler()
train_df[features] = scaler.fit_transform(train_df[features])
test_df[features]  = scaler.transform(test_df[features])

print("StandardScaler applied — train fit, test transformed")

StandardScaler applied — train fit, test transformed


# quality check

In [162]:
print("Train target stats:")
print(train_df['target_4h'].describe())
print(f"\nTest target stats:")
print(test_df['target_4h'].describe())

Train target stats:
count    10579.000000
mean         0.000300
std          0.013089
min         -0.104748
25%         -0.004634
50%          0.000239
75%          0.005185
max          0.146607
Name: target_4h, dtype: float64

Test target stats:
count    2645.000000
mean       -0.000004
std         0.009099
min        -0.050496
25%        -0.004050
50%         0.000005
75%         0.004232
max         0.065513
Name: target_4h, dtype: float64
